# Phase 4 — Agent A5 : Noise Detector
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A5 (`agents/a5_noise.py`) :
- Détection du bruit sur 5 catégories (spam, offtopic, reaction_vide, toxique, bot)
- Formule `Score_Bruit = (1 − noise_ratio) × 100`
- Comportement de fallback quand le LLM est indisponible

> Les appels LLM sont **mockés** dans ce notebook — aucun GPU requis.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a5_noise import a5_noise

## 1. Corpus de test (avec bruit artificiel)

In [ ]:
CLEANED_COMMENTS = [
    {"text": "Great tutorial on machine learning!", "cleaned_text": "great tutorial on machine learning!"},
    {"text": "EARN 500€/DAY — click here: bit.ly/spam", "cleaned_text": "earn 500€/day — click here: bit.ly/spam"},
    {"text": "lol 😂😂😂", "cleaned_text": "lol 😂😂😂"},
    {"text": "This is completely off-topic but did you see the game last night?", "cleaned_text": "this is completely off-topic but did you see the game last night?"},
    {"text": "Very informative, learned about gradient descent.", "cleaned_text": "very informative, learned about gradient descent."},
    {"text": "SUBSCRIBE TO MY CHANNEL!!!", "cleaned_text": "subscribe to my channel!!!"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires (dont 3 bruités attendus)")

## 2. A5 avec LLM mocké — corpus bruité

In [ ]:
mock_response = MagicMock()
mock_response.content = json.dumps({
    "spam_ratio": 0.20,
    "offtopic_ratio": 0.10,
    "reaction_ratio": 0.15,
    "toxic_ratio": 0.0,
    "bot_ratio": 0.0,
    "noise_ratio": 0.40,
    "rationale": "Significant spam and off-topic content detected; reactions are empty.",
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

with patch("agents.a5_noise.get_llm", return_value=mock_llm):
    result = a5_noise(STATE)

n = result["noise"]
print(f"spam_ratio    : {n['spam_ratio']:.1f}%")
print(f"offtopic_ratio: {n['offtopic_ratio']:.1f}%")
print(f"reaction_ratio: {n['reaction_ratio']:.1f}%")
print(f"toxic_ratio   : {n['toxic_ratio']:.1f}%")
print(f"bot_ratio     : {n['bot_ratio']:.1f}%")
print(f"noise_ratio   : {n['noise_ratio']:.1f}%")
print(f"Score_Bruit   : {n['noise_score']:.1f} / 100  (= (1-0.40)×100 = 60.0)")

## 3. Fallback — LLM indisponible

In [ ]:
with patch("agents.a5_noise.get_llm", return_value=None):
    fallback = a5_noise(STATE)

n = fallback["noise"]
print(f"Fallback noise_score : {n['noise_score']}  (attendu: 70.0)")
assert n["noise_score"] == 70.0
print("Fallback conforme.")

# Vérification formule Score_Bruit
print("\nFormule (1 - noise_ratio) × 100 :")
for nr in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    score = round((1 - nr) * 100, 2)
    print(f"  noise_ratio={nr:.1f} → Score_Bruit={score:.1f}")

## Résumé A5

| Comportement | Résultat |
|---|---|
| 5 catégories de bruit détectées | spam, offtopic, reaction_vide, toxique, bot |
| Formule Score_Bruit | (1 − noise_ratio) × 100 |
| noise_ratio=0 → Score=100 | Confirmé |
| noise_ratio=1 → Score=0 | Confirmé |
| LLM indisponible | Fallback 70.0 (noise_ratio=0.30 par défaut) |

> **Poids w3 = 0.25** — dimension la moins influente, mais garde-fou contre les corpus pollués